## tl;dr
Recompute the original frozen 134-run forward-flight cohort using each battery's own middle segment and the v3 Bideal middle slope. There are 585 included drone-trial records, 55 condition rows, and 35 complete five-position rows. The 20 missing position rates are all position 5 without a new B15 calibration. Original telemetry and frozen tables are unchanged.

## Context & Methods
### Key Assumptions
Each battery's middle band is linear. Keep observed forward intervals wholly within its middle band, including a decrease to the lower boundary but not a plateau at that boundary. Retain the original movement mask and through-origin cumulative-drop regression. Multiply each trial's slope by Bideal's middle baseline slope divided by the physical battery's middle baseline slope, then average trial slopes equally within each condition-position. No interval is stretched to 25 seconds.
Source cohort: medium_bideal_v1_20260908. New calibration: individual_three_stage_pooled_candidate_v3_extended100_20260909. Rebuild with: python3 output_py/recompute_medium_rates_v3.py --output <new-directory>.
Legacy QC warnings remain visible. Short segments and integer SOC steps limit rate precision; a flat trace does not mean zero energy use.

## Data
### 1. Load and verify source hashes
This companion verifies the saved results independently from the selected forward-interval evidence. It does not rerun any aircraft code.

In [1]:
from pathlib import Path
import hashlib, json, math
import pandas as pd
root = Path('/Users/alexmason/Downloads/drone_experiment')
out = root / 'analysis_results/medium_forward_bideal_v3_20260909'
manifest = json.loads((out/'manifest.json').read_text())
for name, expected in manifest['input_files_sha256'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
detail = pd.read_csv(out/'run_drone_rates.csv')
points = pd.read_csv(out/'selected_forward_intervals.csv')
wide = pd.read_csv(out/'discharge_rates_wide.csv')
assert not detail.duplicated(['experiment_directory','run_id','drone_name']).any()
print('Input hashes verified:', len(manifest['input_files_sha256']))
print(detail.status.value_counts().to_string())

Input hashes verified: 140
status
included                         585
missing_battery_calibration       48
no_forward_time_in_own_medium     37


## Results
### 2. Recompute all individual slopes and condition means
Zero-valued origin samples do not change the slope numerator or denominator.

In [2]:
key = ['experiment_directory','run_id','drone_name']
rate = 'discharge_rate_Bideal_pp_per_min'
lookup = detail.set_index(key)
checked = 0
for identity, g in points.groupby(key):
    source = lookup.loc[identity]
    x = g.forward_clock_s / 60
    y = g.cumulative_raw_drop_pp * source.physical_to_Bideal_medium_scale
    slope = float((x*y).sum() / (x*x).sum())
    assert math.isclose(slope, source[rate], rel_tol=1e-9, abs_tol=1e-9)
    assert (g.observed_soc_start > source.medium_lower_soc).all()
    assert (g.observed_soc_start <= source.medium_upper_soc).all()
    assert (g.observed_soc_end >= source.medium_lower_soc).all()
    assert (g.observed_soc_end <= source.medium_upper_soc).all()
    checked += 1
assert checked == 585
cells = ['wind_direction','wind_level','formation','inter_drone_spacing_cm']
included = detail[detail.status.eq('included')]
for _, r in wide.iterrows():
    g = included[(included[cells] == r[cells]).all(axis=1)]
    for position in range(1,6):
        values = g[g.position.eq(position)][rate].tolist()
        actual = r[f'position_{position}_{rate}']
        assert (not values and pd.isna(actual)) or math.isclose(math.fsum(values)/len(values), actual, rel_tol=1e-9,abs_tol=1e-9)
print('Individual slopes independently verified:', checked)
print('Condition rows independently verified:', len(wide))
print(wide.head(3).round(3).to_string(index=False))

Individual slopes independently verified: 585
Condition rows independently verified: 55
wind_direction  wind_level formation  inter_drone_spacing_cm  position_1_discharge_rate_Bideal_pp_per_min  position_2_discharge_rate_Bideal_pp_per_min  position_3_discharge_rate_Bideal_pp_per_min  position_4_discharge_rate_Bideal_pp_per_min  position_5_discharge_rate_Bideal_pp_per_min
          head           1    column                      50                                        3.158                                       13.206                                        7.426                                        7.371                                          NaN
          head           1    column                      75                                        7.654                                       13.554                                        7.802                                        4.695                                        7.313
          head           1   diamond                  

## Takeaways
The table uses own-battery medium observations normalized to the Bideal 82–52% middle segment (7.9648164842 pp/min baseline). Of 670 original drone-trial records, 48 lack a B15 calibration and 37 have no forward time in their own middle. There are 85 included segments shorter than 18 s, 17 flat SOC traces, and 129 partial forward segments. These were retained under the existing warning-only policy, so precise configuration rankings require caution. The 20 blank position-5 means are missing, not zero.
Validation: the ordinary Python code cells were executed sequentially and their outputs captured. A Jupyter kernel execution was not tested because nbformat/nbclient are not installed in the available Python environments.